# Decision Tree Regressors — Hitters Salary Prediction

Training, visualising, and pruning regression trees on the Hitters baseball dataset.

**Pipeline:**
1. Load Hitters dataset (322 players, 20 features)
2. Log-transform salary to reduce right-skewness
3. Fit a 5-leaf regression tree on Years + Hits → visualise decision regions
4. Demonstrate overfitting with an unpruned full tree (178 leaves, train MSE ≈ 0)
5. **Cost-complexity pruning** — sweep over α, plot impurity and R², select α*
6. Optimal tree T(α*): 6 leaves, test R² ≈ 0.61 — 172 leaves pruned away

**Dataset:** Hitters (ISLP textbook, Sec. 8.1) — 1986–87 MLB season statistics.  
**Target:** `log(Salary)` — log-transform makes the distribution closer to Gaussian.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_graphviz
from sklearn.metrics import mean_squared_error

plt.style.use('seaborn-v0_8-bright')

---
## 1. Load and Explore

In [ ]:
df = pd.read_csv('data/Hitters.csv')

n_rows            = df.shape[0]
n_features        = df.drop(columns=['Salary']).shape[1]   # exclude the target
missing_counts    = df.isna().sum()
n_with_missing    = (missing_counts > 0).sum()

print(f"Rows                    : {n_rows}")
print(f"Features (excl. Salary) : {n_features}")
print(f"Features with NaNs      : {n_with_missing}")
print()
print("Columns with missing values:")
print(missing_counts[missing_counts > 0])

---
## 2. Log-Transform Salary

Raw salary is right-skewed — most players earn modest amounts with a few high earners pulling the tail. `log(Salary)` compresses the range and produces a distribution much closer to Gaussian, which is better for regression.

Following ISLP Sec. 8.1, features are restricted to `Years` and `Hits` for interpretability and to match the textbook's decision region plots.

In [ ]:
# Drop rows with missing Salary (only feature with NaNs)
df_clean = df.dropna(subset=['Salary']).copy()
df_clean = df_clean[df_clean['Salary'] > 0].copy()   # log requires positive values

print(f"Original: {len(df)} rows → After cleaning: {len(df_clean)} rows")

X        = df_clean[['Years', 'Hits']].copy()
y_salary = df_clean['Salary'].copy()
y        = np.log(y_salary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(y_salary, bins=15, color='steelblue', edgecolor='white')
axes[0].set_title('Salary — Right-skewed')
axes[0].set_xlabel('Salary ($)')
axes[0].set_ylabel('Count')

axes[1].hist(y, bins=15, color='steelblue', edgecolor='white')
axes[1].set_title('log(Salary) — More Gaussian')
axes[1].set_xlabel('log(Salary)')

plt.tight_layout()
plt.show()

---
## 3. Regression Tree with 5 Leaf Nodes

Fitting a constrained tree (max 5 leaves) on all available data — matches the ISLP textbook Figure 8.1.

In [ ]:
reg_tree = DecisionTreeRegressor(max_leaf_nodes=5, random_state=42)
reg_tree.fit(X, y)

print(f"Leaves: {reg_tree.get_n_leaves()}")

In [ ]:
plt.figure(figsize=(14, 7))
plot_tree(reg_tree, feature_names=list(X.columns),
          filled=True, rounded=True, fontsize=10)
plt.title('Regression Tree (max_leaf_nodes = 5)')
plt.tight_layout()
plt.show()

---
## 4. Decision Regions in Feature Space

Plotting the tree's rectangular decision regions in (Years, Hits) space. Each colour corresponds to one leaf node — the predicted log(Salary) value for that region.

In [ ]:
x_min, x_max = X['Years'].min() - 1, X['Years'].max() + 1
y_min, y_max = X['Hits'].min() - 10, X['Hits'].max() + 10

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 400),
    np.linspace(y_min, y_max, 400)
)
grid_df = pd.DataFrame({'Years': xx.ravel(), 'Hits': yy.ravel()})
zz = reg_tree.predict(grid_df).reshape(xx.shape)

plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, zz, alpha=0.35)
plt.scatter(X['Years'], X['Hits'], s=20, edgecolor='k', alpha=0.7)
plt.xlabel('Years')
plt.ylabel('Hits')
plt.title('Decision Regions (predicting log(Salary))')
plt.tight_layout()
plt.show()

In [ ]:
# Verify: points per leaf should match tree's 'samples' values
leaf_ids = reg_tree.apply(X)
unique_leaves, counts = np.unique(leaf_ids, return_counts=True)

print("Points per decision region (leaf_id → count):")
for lid, c in zip(unique_leaves, counts):
    print(f"  leaf {lid} → {c} samples")
print()
print("These match the 'samples' field in each leaf node of the tree diagram.")

---
## 5. Overfitting: The Full (Unpruned) Tree

Training a tree with no constraints shows classic overfitting — the tree memorises the training data perfectly (train MSE ≈ 0) but generalises poorly.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
print(f"Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples")

In [ ]:
full_tree = DecisionTreeRegressor(random_state=42)
full_tree.fit(X_train, y_train)

train_mse = mean_squared_error(y_train, full_tree.predict(X_train))
test_mse  = mean_squared_error(y_test,  full_tree.predict(X_test))

print(f"Leaves          : {full_tree.get_n_leaves()}")
print(f"Train MSE       : {train_mse:.6f}")
print(f"Test MSE        : {test_mse:.4f}")
print(f"Train/Test ratio: {test_mse / train_mse:.0f}×")
print()
print("""
Overfitting is clear: train MSE ≈ 0 (tree memorises every training point)
while test MSE ≈ 0.48 — nearly 400× larger. The tree has grown 178 leaves
for 184 training samples, creating near-one-sample leaf nodes.
""")

---
## 6. Cost-Complexity Pruning

The regularised loss is:
$$R_\alpha(T) = R(T) + \alpha |T|$$
where $R(T)$ is the total leaf impurity, $|T|$ is the number of leaves, and $\alpha \geq 0$ is a penalty on tree complexity.

**`cost_complexity_pruning_path()`** traces the sequence of subtrees $T_\alpha$ as $\alpha$ increases from 0 (full tree) to ∞ (single-leaf tree), returning all candidate $\alpha$ values and their total leaf impurities.

### 6.1 Compute pruning path

In [ ]:
path       = full_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas
impurities = path.impurities

print(f"Number of α candidates : {len(ccp_alphas)}")
print(f"α range                : [{ccp_alphas[0]:.6f}, {ccp_alphas[-1]:.4f}]")
print(f"Last tree (max α)      : {len(ccp_alphas)-1} → 1 leaf (trivial tree)")

### 6.2 Total leaf impurity vs α

As α increases, the tree is pruned more aggressively — fewer leaves, larger regions, higher impurity. The step-like shape reflects that tree structure only changes at discrete α thresholds.

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(ccp_alphas, impurities, marker='o', drawstyle='steps-post')
plt.xlabel('α (ccp_alpha)')
plt.ylabel('Total leaf impurity')
plt.title('Total Leaf Impurity vs α')
plt.tight_layout()
plt.show()

### 6.3 Train one tree per α value

In [ ]:
trees = []
for a in ccp_alphas:
    t = DecisionTreeRegressor(random_state=42, ccp_alpha=a)
    t.fit(X_train, y_train)
    trees.append(t)

last_tree  = trees[-1]
last_alpha = ccp_alphas[-1]
print(f"Last α = {last_alpha:.4f}  →  {last_tree.get_n_leaves()} leaf  (trivial predictor)")

### 6.4 R² vs α — selecting α*

Plotting train and test R² for every pruned tree. The optimal α* is chosen as the value that maximises test R² — it balances model complexity against generalisation.

In [ ]:
train_scores = [t.score(X_train, y_train) for t in trees]
test_scores  = [t.score(X_test,  y_test)  for t in trees]

plt.figure(figsize=(7, 5))
plt.plot(ccp_alphas, train_scores, marker='o', label='train', drawstyle='steps-post')
plt.plot(ccp_alphas, test_scores,  marker='o', label='test',  drawstyle='steps-post')
plt.xlabel('α (ccp_alpha)')
plt.ylabel('Score (R²)')
plt.title('Train and Test R² vs α')
plt.legend()
plt.tight_layout()
plt.show()

print("""
Pattern: as α increases from 0, train R² drops steadily (less overfitting).
Test R² first rises (underfitting reduced), then falls (model too simple).
The peak of the test curve marks α* — the optimal regularisation strength.
""")

### 6.5 Optimal tree T(α*)

In [ ]:
best_idx   = int(np.argmax(test_scores))
alpha_star = ccp_alphas[best_idx]
best_tree  = trees[best_idx]

print(f"α*                   : {alpha_star:.6f}")
print(f"Test R² at α*        : {test_scores[best_idx]:.4f}")
print(f"Train R² at α*       : {train_scores[best_idx]:.4f}")
print(f"Leaves in T(α*)      : {best_tree.get_n_leaves()}")
print(f"Leaves in full tree  : {full_tree.get_n_leaves()}")
print(f"Reduction            : {full_tree.get_n_leaves() - best_tree.get_n_leaves()} leaves pruned")
print()
print("""
Pruning reduced the tree from 178 leaves to 6 — eliminating 172 splits
that were fitting noise rather than signal. Test R² improved from near 0
(with the full tree's overfitting) to ~0.61 with T(α*).
""")

In [ ]:
plt.figure(figsize=(14, 7))
plot_tree(best_tree, feature_names=list(X.columns),
          filled=True, rounded=True, fontsize=10)
plt.title(f'Optimal Pruned Tree T(α*) — α* = {alpha_star:.4f}, {best_tree.get_n_leaves()} leaves')
plt.tight_layout()
plt.show()

---
## Summary

| Model | Leaves | Train MSE | Test R² | Notes |
|---|---|---|---|---|
| 5-leaf tree (all data) | 5 | — | — | Interpretable baseline |
| Full unpruned tree | 178 | ≈0.001 | ≈0.00 | Severe overfitting |
| T(α*) pruned | 6 | — | ≈0.61 | Optimal generalisation |

Key insights:
- `Years ≤ 4.5` is the dominant first split — experience is the strongest salary predictor
- `Hits` provides the secondary discrimination within the experienced-player group
- Cost-complexity pruning is an elegant way to automatically select tree complexity using held-out data rather than manual hyperparameter search